In [1]:
# QueryLLM orchestrator: wires every agent into ONE analyst-facing function.
# Data flow (same isolation as notebooks 03-05):
#   catalog.yaml + rules.yaml (files) -> prompts -> LLM -> SQL text
#   -> the notebook itself runs the SQL read-only -> answer
import os
import re
import json
import time
from pathlib import Path

import requests
import yaml
from dotenv import load_dotenv
import oracledb

# project root = parent of notebook/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

# opencode inference endpoint (OpenAI-compatible chat completions)
LLM_URL = os.getenv("LLM_URL")
LLM_KEY = os.getenv("SQL_AGENT")     # API key stored under the name SQL_AGENT
LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.8-flash")

missing = [n for n, v in {"LLM_URL": LLM_URL, "SQL_AGENT": LLM_KEY}.items() if not v]
if missing:
    raise ValueError(f"Missing variables in .env: {', '.join(missing)}")

CATALOG_PATH = PROJECT_ROOT / "catalog" / "catalog.yaml"
RULES_PATH = PROJECT_ROOT / "catalog" / "rules.yaml"
print(f"model: {LLM_MODEL} | catalog: {CATALOG_PATH.name} | rules: {RULES_PATH.name}")

model: qwen3.8-flash | catalog: catalog.yaml | rules: rules.yaml


In [2]:
# Every agent function is DEFINED in notebooks 03/04/05 and exported to src/
# by their export cells. This notebook keeps only what it defines first:
# explanation_agent() and query_llm() (exported below to src/orchestrator.py).
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.llm import llm
from src.intent import intent_agent
from src.writer import sql_writer
from src.validator import validator_agent

print("agent stack imported from src/ (defined first in notebooks 03/04/05)")

agent stack imported from src/ (defined first in notebooks 03/04/05)


In [3]:
# Load the catalog and the PDP rules. The rules text is rendered inside
# sql_writer/repair_sql (src modules), so nothing to prepare here.
catalog = yaml.safe_load(CATALOG_PATH.read_text(encoding="utf-8"))
rules = yaml.safe_load(RULES_PATH.read_text(encoding="utf-8"))

print("policy:", catalog["policy"])
print(f"rules: v{rules['rules_version']}")

policy: {'max_workspaces_per_question': 1, 'cross_schema_joins': 'disallowed', 'cross_schema_fk_count': 0, 'connection_count': 3}
rules: v1.2


In [4]:
# Explanation agent: plain-language summary of the final SQL for the
# analyst (and the DE reviewing it). It states which columns were
# anonymized, per the PDP rules.
def explanation_agent(question, sql):
    prompt = (
        "Explain what this SQL query does in 2-3 plain sentences for a data "
        "analyst. Mention any column that appears anonymized/masked or "
        "aggregated in the query.\n\n"
        f"Question: {question}\n\n"
        f"SQL:\n{sql}\n\n"
        "Explanation:"
    )
    return llm(prompt, system="You explain SQL queries simply.", session="ses_queryllm_explain")

In [5]:
# query_llm: the ONE analyst-facing function. Question in, answer out -
# the analyst never sees workspaces, schemas or table names.
# Steps: intent -> SQL writer -> validator (repair loop) -> explanation.\n# catalog/rules are parameters, not globals: exports verbatim to src/orchestrator.py.
def query_llm(question, catalog, rules, max_retries=3, explain=True):
    # per-stage durations are recorded in the result (and the logs) so every
    # speed optimization can be measured before/after
    t_start = time.perf_counter()
    workspaces = intent_agent(question, catalog)
    t_intent = time.perf_counter() - t_start
    if workspaces is None:
        # honest refusal: the question is outside every catalog workspace
        return {
            "question": question,
            "workspaces": None,
            "status": "no_route",
            "answer": "This question cannot be answered with the data available in the catalog.",
        }

    print(f"routed to: {workspaces}")
    t_write_start = time.perf_counter()
    draft = sql_writer(question, workspaces, catalog, rules)
    t_write = time.perf_counter() - t_write_start
    t_validate_start = time.perf_counter()
    outcome = validator_agent(question, draft, workspaces, catalog, rules, max_retries=max_retries)
    t_validate = time.perf_counter() - t_validate_start

    durations = {
        "intent_s": round(t_intent, 2),
        "write_s": round(t_write, 2),
        "validate_s": round(t_validate, 2),
        "total_s": round(time.perf_counter() - t_start, 2),
    }

    if outcome["status"] != "ok":
        return {
            "question": question,
            "workspaces": workspaces,
            "status": "failed",
            "sql": outcome["sql"],
            "history": outcome["history"],
            "durations": durations,
        }

    result = {
        "question": question,
        "workspaces": workspaces,
        "status": "ok",
        "sql": outcome["sql"],
        "columns": outcome["columns"],
        "rows": outcome["rows"],
        "attempts": outcome["attempts"],
        "history": outcome["history"],
        "durations": durations,
    }
    # the dashboard skips the explanation (explain=False): it shows the SQL
    # only, so no explanation LLM call is spent
    if explain:
        result["explanation"] = explanation_agent(question, outcome["sql"])
    return result

In [6]:
# End-to-end demo: four analyst questions covering every path:
# single workspace, two workspaces (candidate connection), PDP anonymization
# and a question outside the catalog (honest no-route answer).
QUESTIONS = [
    "How many employees are in each department?",
    "Compare the countries listed in the HR system with the countries where we have sales",
    "Show customer names and their cities from the sales data",
    "How many servers do we have in the data center?",
]

results = {}
for question in QUESTIONS:
    print(f"\n=== {question}")
    result = query_llm(question, catalog, rules)
    results[question] = result

    if result["status"] in ("no_route", "failed"):
        print(f"status: {result['status']}")
        print(f"answer: {result.get('answer', '(see history)')}")
        continue

    print(f"workspaces: {result['workspaces']} | attempts: {result['attempts']}")
    print("SQL:")
    print(result["sql"])
    print("preview:")
    for row in result["rows"][:3]:
        print("  ", row)
    print("explanation:")
    print(f"  {result['explanation']}")


=== How many employees are in each department?


routed to: ['HR']


workspaces: ['HR'] | attempts: 1
SQL:
SELECT d.DEPARTMENT_ID,
       d.DEPARTMENT_NAME,
       COUNT(e.EMPLOYEE_ID) AS EMPLOYEE_COUNT
FROM HR.DEPARTMENTS d
LEFT JOIN HR.EMPLOYEES e
  ON e.DEPARTMENT_ID = d.DEPARTMENT_ID
GROUP BY d.DEPARTMENT_ID,
         d.DEPARTMENT_NAME
ORDER BY EMPLOYEE_COUNT DESC
preview:
   (50, 'Shipping', 45)
   (80, 'Sales', 34)
   (30, 'Purchasing', 6)
explanation:
  This query counts how many employees are assigned to each department, including departments with zero employees. It returns each department’s ID and name along with the aggregated column `EMPLOYEE_COUNT`, which is calculated using `COUNT(e.EMPLOYEE_ID)`. No columns appear anonymized or masked in the query.

=== Compare the countries listed in the HR system with the countries where we have sales


status: no_route
answer: This question cannot be answered with the data available in the catalog.

=== Show customer names and their cities from the sales data


routed to: ['SH']


workspaces: ['SH'] | attempts: 1
SQL:
WITH MASKED_CUSTOMERS AS (
  SELECT
    CUST_ID,
    CASE
      WHEN CUST_FIRST_NAME IS NOT NULL
      THEN SUBSTR(TO_CHAR(CUST_FIRST_NAME), 1, 1) || '***_' ||
           SUBSTR(RAWTOHEX(STANDARD_HASH(LOWER(TRIM(TO_CHAR(CUST_FIRST_NAME))), 'SHA256')), 1, 8)
    END AS CUST_FIRST_NAME,
    CASE
      WHEN CUST_LAST_NAME IS NOT NULL
      THEN SUBSTR(TO_CHAR(CUST_LAST_NAME), 1, 1) || '***_' ||
           SUBSTR(RAWTOHEX(STANDARD_HASH(LOWER(TRIM(TO_CHAR(CUST_LAST_NAME))), 'SHA256')), 1, 8)
    END AS CUST_LAST_NAME,
    CUST_CITY
  FROM SH.CUSTOMERS
)
SELECT
  m.CUST_FIRST_NAME,
  m.CUST_LAST_NAME,
  m.CUST_CITY
FROM SH.SALES s
JOIN MASKED_CUSTOMERS m
  ON s.CUST_ID = m.CUST_ID
GROUP BY
  m.CUST_FIRST_NAME,
  m.CUST_LAST_NAME,
  m.CUST_CITY
preview:
   ('A***_018E7458', 'E***_0F1324DE', 'Mackville')
   ('A***_018E7458', 'L***_929DF7FC', 'Dolores')
   ('A***_018E7458', 'P***_A41A6284', 'Montara')
explanation:
  It returns unique customer names and citi

status: no_route
answer: This question cannot be answered with the data available in the catalog.


In [7]:
# Summary of the full pipeline run
ok = sum(1 for r in results.values() if r["status"] == "ok")
failed = sum(1 for r in results.values() if r["status"] == "failed")
no_route = sum(1 for r in results.values() if r["status"] == "no_route")
print(f"answered: {ok} | failed: {failed} | no route: {no_route} | total: {len(results)}")
for question, r in results.items():
    detail = ""
    if r["status"] == "ok":
        detail = f"{r['workspaces']} attempts={r['attempts']}"
    elif r["status"] == "failed":
        detail = "; ".join(h["outcome"][:40] for h in r["history"])
    print(f"  [{r['status']:<8}] {question} {detail}")

answered: 2 | failed: 0 | no route: 2 | total: 4
  [ok      ] How many employees are in each department? ['HR'] attempts=1
  [no_route] Compare the countries listed in the HR system with the countries where we have sales 
  [ok      ] Show customer names and their cities from the sales data ['SH'] attempts=1
  [no_route] How many servers do we have in the data center? 


In [8]:
# Export to src/: the functions defined above in this notebook (first
# definitions) are written to src/orchestrator.py so later notebooks import
# them instead of copying. Re-run this notebook after changing them.
import inspect

SRC_DIR = PROJECT_ROOT / "src"
def export_module(filename, header, names):
    parts = [header]
    for name in names:
        parts.append("\n\n" + inspect.getsource(globals()[name]).rstrip() + "\n")
    (SRC_DIR / filename).write_text("".join(parts), encoding="utf-8")
    print(f"exported src/{filename}: {', '.join(names)}")

export_module(
    "orchestrator.py",
    "# Orchestrator (first defined in notebook/06_queryllm.ipynb).\n"
    "# query_llm(question, catalog, rules) is THE analyst-facing entry point:\n"
    "# intent -> SQL writer -> validator (repair loop) -> explanation.\n"
    "import time\n\n"
    "from src.llm import llm\n"
    "from src.intent import intent_agent\n"
    "from src.writer import sql_writer\n"
    "from src.validator import validator_agent\n",
    ["explanation_agent", "query_llm"],
)

exported src/orchestrator.py: explanation_agent, query_llm
